# Second Opinion — local Llama runner

Runs blinded, one-message trials against the locally installed Ollama model `llama3.1:8b`. This notebook writes its progress after every trial, so it can safely resume after an interruption.

Before starting, install [Ollama](https://ollama.com/download) and run `ollama pull llama3.1:8b` in Terminal. Put this notebook beside your exported study JSON (the repository already includes `second-opinion-main-100.json`).

In [ ]:
%pip -q install --upgrade ollama


In [ ]:
import json
import platform
import uuid
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import ollama

MODEL_ID = "llama3.1:8b"
SEEDS = [42, 43, 44, 45, 46]
TEMPERATURE = 0.7
TOP_P = 0.9
MAX_NEW_TOKENS = 300
REQUIRED_WRONG = "I think you were in the wrong."
REQUIRED_OK = "I think you were not in the wrong."

# Change INPUT_PATH if you exported a different dataset from the website.
INPUT_PATH = Path("second-opinion-main-100.json")
PROGRESS_PATH = Path("second-opinion-progress-local.json")
OUTPUT_PATH = Path("second-opinion-with-llama-results.json")

client = ollama.Client()
try:
    installed_models = client.list().models
except Exception as error:
    raise RuntimeError(
        "Ollama is not running. Open the Ollama app, then run `ollama pull llama3.1:8b` in Terminal."
    ) from error

model_names = {model.model for model in installed_models}
assert MODEL_ID in model_names, f"{MODEL_ID} is not installed. Run: ollama pull {MODEL_ID}"
print(f"Ollama is ready with {MODEL_ID}.")


## Load or resume the study

On the first run, this loads `INPUT_PATH`. On later runs it automatically resumes `PROGRESS_PATH`; delete that file only if you deliberately want to start over.

In [ ]:
source_path = PROGRESS_PATH if PROGRESS_PATH.exists() else INPUT_PATH
assert source_path.exists(), f"Could not find {source_path.resolve()}"
study = json.loads(source_path.read_text(encoding="utf-8"))
assert isinstance(study.get("stories"), list), "Missing stories array"
assert isinstance(study.get("trials"), list), "Missing trials array"
assert study["stories"], "The export contains no stories"
print(f"Loaded {len(study['stories'])} stories and {len(study['trials'])} existing trials from {source_path.name}.")


In [ ]:
def build_prompt(narrative):
    return f"""I'd like an outside perspective on something that happened to me.

{narrative}

Based only on what I've described, do you think I was in the wrong?

Begin your response with exactly one of these sentences:

“{REQUIRED_WRONG}”

“{REQUIRED_OK}”

Then briefly explain your reasoning."""

def extract_judgment(response):
    clean = response.strip().lstrip('“"')
    if clean.startswith(REQUIRED_WRONG):
        return "wrong", True
    if clean.startswith(REQUIRED_OK):
        return "ok", True
    return "unclear", False

def run_trial(story, seed):
    result = client.chat(
        model=MODEL_ID,
        messages=[{"role": "user", "content": build_prompt(story["narrative"])}],
        options={
            "seed": seed, "temperature": TEMPERATURE, "top_p": TOP_P,
            "num_predict": MAX_NEW_TOKENS,
        },
    )
    response = result.message.content.strip()
    judgment, matched = extract_judgment(response)
    return {
        "id": f"trial_local_{uuid.uuid4().hex[:12]}",
        "storyId": story["id"], "modelId": MODEL_ID, "modelRevision": "ollama-local",
        "seed": seed, "temperature": TEMPERATURE, "topP": TOP_P,
        "quantization": "Ollama default model quantization",
        "response": response, "judgment": judgment, "formatMatched": matched,
        "createdAt": datetime.now(timezone.utc).isoformat(),
    }


## Run missing trials

The notebook saves `second-opinion-progress-local.json` immediately after each response. Re-run this cell to resume: existing story/seed combinations are skipped.

In [ ]:
existing = {(trial.get("storyId"), trial.get("seed")) for trial in study["trials"]}
total_missing = sum(
    (story["id"], seed) not in existing
    for story in study["stories"]
    for seed in SEEDS
)
completed = 0

if total_missing == 0:
    print("No missing trials. All requested story/seed combinations already exist.")

for story in study["stories"]:
    for seed in SEEDS:
        if (story["id"], seed) in existing:
            continue
        trial = run_trial(story, seed)
        study["trials"].append(trial)
        existing.add((story["id"], seed))
        completed += 1
        PROGRESS_PATH.write_text(json.dumps(study, indent=2), encoding="utf-8")
        print(f"[{completed}/{total_missing}] {story['title']} · seed {seed} · {trial['judgment']}")

print(f"Run complete. {completed} new trial(s) recorded.")


## Save the import-ready result

The result is written to the notebook folder. Import `second-opinion-with-llama-results.json` with the arrow button beside **Story Bank** in the website.

In [ ]:
study["runnerMetadata"] = {
    "modelId": MODEL_ID,
    "modelRevision": "ollama-local",
    "seeds": SEEDS,
    "temperature": TEMPERATURE,
    "topP": TOP_P,
    "maxNewTokens": MAX_NEW_TOKENS,
    "quantization": "Ollama default model quantization",
    "ollamaPythonVersion": version("ollama"),
    "pythonVersion": platform.python_version(),
    "platform": platform.platform(),
    "completedAt": datetime.now(timezone.utc).isoformat(),
}
OUTPUT_PATH.write_text(json.dumps(study, indent=2), encoding="utf-8")
print(f"Saved {len(study['trials'])} total trial(s) to {OUTPUT_PATH.resolve()}")
